# 03 · Inferência na GPU (nuvem): MedGemma 27B

Só o modelo que não cabe no Mac (MedGemma 27B, ~54 GB em bf16). Os outros três rodam no Mac (`03b_inferencia_local`).
Etapas (`vlmdr.runner`):
0. **checagem no piloto**: 20 imagens por base (`run = piloto_cuda`, fora das análises). Se houver P(referenciável) `nan`,
   mais de 10% de respostas inválidas ou massa média nos graus < 0,9, o modelo é pulado e o motivo vai para o log;
1. prompt principal (`main_en`) nas **duas bases primeiro** (desfecho primário);
2. prompts alternativos (`nodef_en`, `main_pt`) na subamostra de robustez;
3. segunda execução do prompt principal na subamostra de reprodutibilidade (`run = r2`).

Grava a cada imagem e retoma de onde parou. **Rode pelo `rodar_nuvem.sh`** (veja o README): ele executa este notebook
sem depender do navegador, para no limite `MAX_HORAS`, empacota os resultados em `resultados_nuvem.tar.gz` e para a
máquina pela API da GPU Brasil (o disco é mantido).

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

import config
pd.set_option("display.max_columns", 50, "display.width", 250, "display.max_rows", 200)
from vlmdr import cohort, infer, prompts
from vlmdr.runner import Runner

In [ ]:
import os, shutil, subprocess, tarfile, traceback
import torch, transformers
from huggingface_hub import constants as hf_constants, whoami

MODELOS = os.environ.get("MODELOS", "medgemma_27b").split(",")
LIMITE = int(os.environ["LIMITE"]) if os.environ.get("LIMITE") else None   # ex.: 5 para um ensaio rápido
ENCERRAR = os.environ.get("ENCERRAR", "1") == "1"   # o rodar_nuvem.sh põe 0 e para a máquina ele mesmo
ENCERRAR_CMD = os.environ.get("ENCERRAR_CMD", "python3 ../gpubrasil.py parar")   # shutdown NÃO para a cobrança

assert torch.cuda.is_available(), "sem GPU CUDA: este notebook é só para a máquina da nuvem"
print("GPU:", torch.cuda.get_device_name(0), "| VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9))
print("torch", torch.__version__, "| CUDA", torch.version.cuda, "| transformers", transformers.__version__)
print("disco livre (GB):", round(shutil.disk_usage(config.ROOT).free / 1e9), "| cache dos modelos:", hf_constants.HF_HUB_CACHE)
print("Hugging Face:", whoami()["name"], "| modelos:", MODELOS, "| limite:", LIMITE)
cohorts = {ds: cohort.load(ds) for ds in ("brset", "mbrset")}
for ds, c in cohorts.items():
    falta = int((~c.image_path.map(lambda p: Path(p).exists())).sum())
    print(f"{ds}: {len(c)} imagens, {falta} não encontradas")
    assert falta == 0, f"imagens do {ds} não encontradas: confira {ds.upper()}_IMAGES"
runner = Runner(cohorts, "gpu", limite=LIMITE, limpar_cache=len(MODELOS) > 1)

## Execução (empacota os resultados e para a máquina no fim, mesmo se der erro)

In [ ]:
try:
    for m in MODELOS:
        try:
            runner.roda(m)
        except Exception as e:
            runner.registra(modelo=m, etapa="erro", problemas=f"{type(e).__name__}: {e}")
            traceback.print_exc()
finally:
    destino = config.ROOT / "resultados_nuvem.tar.gz"
    with tarfile.open(destino, "w:gz") as tar:
        tar.add(config.RESULTS, arcname="results")
    print("resultados em", destino, flush=True)
    if ENCERRAR:
        print("parando a máquina:", ENCERRAR_CMD, flush=True)
        subprocess.run(ENCERRAR_CMD, shell=True)
pd.DataFrame(runner.log)